In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sqlalchemy import text
from src import config as C
from src.db_utils import run_sql_file

run_sql_file("03_features.sql", {
    "first_week": C.FIRST_WEEK, "sim_end": C.SIM_END, "train_end": C.TRAIN_END,
})

engine = C.get_engine()
feat = pd.read_sql(text("SELECT * FROM features_weekly"), engine, parse_dates=["week_start"])
print(feat.shape)
feat.head()

(130000, 13)


,store,item,week_start,units,lag_1,lag_2,lag_4,lag_52,roll4_mean,roll13_mean,week_of_year,month_num,data_split
0,1,1,2013-01-07,66,NaN,NaN,NaN,NaN,NaN,NaN,2,1,train
1,1,1,2013-01-14,80,66.0,NaN,NaN,NaN,66.0,66.0,3,1,train
2,1,1,2013-01-21,70,80.0,66.0,NaN,NaN,73.0,73.0,4,1,train
3,1,1,2013-01-28,86,70.0,80.0,NaN,NaN,72.0,72.0,5,1,train
4,1,1,2013-02-04,82,86.0,70.0,66.0,NaN,75.5,75.5,6,2,train


In [2]:
print(feat["data_split"].value_counts(), "\n")
print(feat.isna().sum(), "\n")

# Leakage test on one series: features must equal values from EARLIER weeks
s = feat[(feat.store == 1) & (feat.item == 1)].sort_values("week_start").reset_index(drop=True)
print("lag_1 correct:  ", (s["lag_1"].iloc[1:] == s["units"].shift(1).iloc[1:]).all())
print("lag_52 correct: ", (s["lag_52"].iloc[52:] == s["units"].shift(52).iloc[52:]).all())
manual = s["units"].shift(1).rolling(4).mean()
print("roll4 correct:  ", ((s["roll4_mean"] - manual).abs().iloc[4:] < 1e-9).all())

data_split
train    104000
test      26000
Name: count, dtype: int64 

store               0
item                0
week_start          0
units               0
lag_1             500
lag_2            1000
lag_4            2000
lag_52          26000
roll4_mean        500
roll13_mean       500
week_of_year        0
month_num           0
data_split          0
dtype: int64 

lag_1 correct:   True
lag_52 correct:  True
roll4 correct:   True
